In [1]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image
import os

In [2]:
# Load my image
image_path = "raw.jpeg"

if os.path.exists(image_path):
    print("Image found!")
    image = Image.open(image_path)
    width, height = image.size
    print(f"Image dimensions: {width} x {height} p")
else:
    print("Image not found!")


Image found!
Image dimensions: 1024 x 768 p


#### YOLOv8 Model Sizes:
- **yolov8n.pt**: Nano - Fastest, least accurate
- **yolov8s.pt**: Small - Good balance
- **yolov8m.pt**: Medium - Better accuracy
- **yolov8l.pt**: Large - High accuracy
- **yolov8x.pt**: Extra Large - Highest accuracy, slowest

In [3]:
# Load YOLO v8 model
model = YOLO('yolov8m.pt')
results = model(image_path)

# Detection results (confidence levels)
print("\nDetection Results:")

detection_count = 0
for result in results:
    boxes = result.boxes
    if boxes is not None:
        for box in boxes:
            confidence = box.conf[0].item()
            if confidence > 0.5:  # Only show confident detections
                detection_count += 1
                class_id = int(box.cls[0].item())
                class_name = result.names[class_id]
                print(f"{class_name}: {confidence:.2f} confidence")
    else:
        print("No objects detected")

print(f"\nTotal objects detected: {detection_count}")


image 1/1 /Users/nghiatran/projects/ai-ml/ml/object-detection/raw.jpeg: 480x640 3 persons, 4 tvs, 1 laptop, 1 keyboard, 240.2ms
Speed: 2.3ms preprocess, 240.2ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

Detection Results:
person: 0.95 confidence
person: 0.88 confidence
tv: 0.79 confidence
laptop: 0.78 confidence
keyboard: 0.68 confidence
tv: 0.63 confidence

Total objects detected: 6


In [4]:
# Load image for visualization
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Create the plot
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.imshow(image_rgb)

# Draw bounding boxes and labels
detection_count = 0
for result in results:
    boxes = result.boxes
    if boxes is not None:
        for box in boxes:
            confidence = box.conf[0].item()
            if confidence > 0.5:
                detection_count += 1
                
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                
                # Draw rectangle
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                       linewidth=3, edgecolor='red', facecolor='none')
                ax.add_patch(rect)
                
                # Add label
                class_id = int(box.cls[0].item())
                class_name = result.names[class_id]
                label = f"{class_name}: {confidence:.2f}"
                ax.text(x1, y1-15, label, fontsize=12, color='red', fontweight='bold',
                       bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.9))

ax.set_title(f"YOLOv8 Object Detection Results\nFound {detection_count} objects", fontsize=16, fontweight='bold')
ax.axis('off')
plt.tight_layout()

# Save the result as detect.jpg
plt.savefig('detect.jpg', dpi=150, bbox_inches='tight')
print(f"Total objects detected: {detection_count}")

plt.close()

Total objects detected: 6


In [5]:
# Try different confidence thresholds
confidence_thresholds = [0.3, 0.5, 0.7]

for threshold in confidence_thresholds:
    print(f"\nDetection with confidence threshold: {threshold}")
    
    detection_count = 0
    for result in results:
        boxes = result.boxes
        if boxes is not None:
            for box in boxes:
                confidence = box.conf[0].item()
                if confidence > threshold:
                    detection_count += 1
                    class_id = int(box.cls[0].item())
                    class_name = result.names[class_id]
                    print(f"  • {class_name}: {confidence:.2f}")
    
    print(f"Total objects detected: {detection_count}")



Detection with confidence threshold: 0.3
  • person: 0.95
  • person: 0.88
  • tv: 0.79
  • laptop: 0.78
  • keyboard: 0.68
  • tv: 0.63
  • tv: 0.46
  • person: 0.41
  • tv: 0.32
Total objects detected: 9

Detection with confidence threshold: 0.5
  • person: 0.95
  • person: 0.88
  • tv: 0.79
  • laptop: 0.78
  • keyboard: 0.68
  • tv: 0.63
Total objects detected: 6

Detection with confidence threshold: 0.7
  • person: 0.95
  • person: 0.88
  • tv: 0.79
  • laptop: 0.78
Total objects detected: 4
